In [1]:
#################################
#################################
from scripts.preprocessing import simplify_swc_topology

import matplotlib.lines as mlines
import matplotlib.pyplot as plt
from typing import Union
import networkx as nx
import pandas as pd
import sys
import os 

In [ ]:
#######################################
#######################################
def plot_neuron_topology(data: Union[str, pd.DataFrame], 
                         labels_df: pd.DataFrame = None, 
                         draw_node_id: bool = True):
    """
    Parses an SWC file or DataFrame and plots a customized hierarchical topological tree.
    
    Parameters:
    - data (str or pd.DataFrame): The string path to the .swc file OR a DataFrame representing the SWC.
    - labels_df (pd.DataFrame, optional): A DataFrame containing labels. 
      Assumes column 0 is the Node ID and column 1 is the label text. Defaults to None.
    - draw_node_id (bool, optional): Whether to display the numerical node ID next to the node. Defaults to True.
    """
    
    # ---------------------------------------------------------
    # 1. Parse SWC & Build Directed Graph
    # ---------------------------------------------------------
    G = nx.DiGraph()
    node_types = {} 

    if isinstance(data, pd.DataFrame):
        cols = [str(c).lower() for c in data.columns]
        
        id_col = data.columns[cols.index('id')] if 'id' in cols else data.columns[0]
        parent_col = data.columns[cols.index('parent')] if 'parent' in cols else data.columns[6]
        type_col = data.columns[cols.index('type')] if 'type' in cols else (data.columns[1] if len(data.columns) > 1 else None)

        for _, row in data.iterrows():
            node_id = int(row[id_col])
            parent_id = int(row[parent_col])
            
            G.add_node(node_id) 
            if parent_id != -1: 
                G.add_edge(parent_id, node_id)
                
            if type_col is not None:
                node_types[node_id] = row[type_col]

    elif isinstance(data, str):
        try:
            with open(data, "r") as f:
                for line in f:
                    if line.startswith("#") or not line.strip():
                        continue
                    parts = line.split()
                    if len(parts) >= 7:
                        node_id = int(parts[0])
                        node_type = parts[1]  
                        parent_id = int(parts[6])
                        
                        G.add_node(node_id) 
                        if parent_id != -1: 
                            G.add_edge(parent_id, node_id)
                            
                        node_types[node_id] = node_type
        except FileNotFoundError:
            print(f"Error: Could not find the file at {data}")
            return
    else:
        print("Error: Input data must be a file path string or a pandas DataFrame.")
        return
        
    if not G.nodes():
        print("Error: No valid nodes were found.")
        return

    # ---------------------------------------------------------
    # 2. Process Labels and Node Types
    # ---------------------------------------------------------
    label_dict = {}
    if labels_df is not None and not labels_df.empty:
        id_col = labels_df.columns[0]
        val_col = labels_df.columns[1]
        label_dict = dict(zip(labels_df[id_col], labels_df[val_col]))
        
        cols_lower = [str(c).lower() for c in labels_df.columns]
        if 'type' in cols_lower:
            type_col = labels_df.columns[cols_lower.index('type')]
            type_dict = dict(zip(labels_df[id_col], labels_df[type_col]))
            node_types.update(type_dict)

    # ---------------------------------------------------------
    # 3. Leaf-Weighted Hierarchical Layout Algorithm (Iterative)
    # ---------------------------------------------------------
    # 3a. Calculate leaf counts for all nodes bottom-up in one pass
    leaf_counts = {}
    out_degree = dict(G.out_degree())
    queue = [n for n in G.nodes() if out_degree[n] == 0]
    processed_children = {n: 0 for n in G.nodes()}
    
    while queue:
        curr = queue.pop(0)
        if out_degree[curr] == 0:
            leaf_counts[curr] = 1
            
        for p in G.predecessors(curr):
            processed_children[p] += 1
            # Add current child's leaf count to the parent
            leaf_counts[p] = leaf_counts.get(p, 0) + leaf_counts.get(curr, 1)
            # Once all children of a parent are processed, queue the parent
            if processed_children[p] == out_degree[p]:
                queue.append(p)

    # 3b. Assign spatial coordinates iteratively using an explicit stack
    def get_hierarchy_pos_iterative(G, root, xcenter=0.5, vert_gap=1.0, vert_loc=0):
        pos = {}
        total_tree_leaves = leaf_counts.get(root, 1)
        available_width = max(1.0, total_tree_leaves * 1.0)
        
        # Stack stores layout tuples: (node, xcenter, vert_loc, available_width)
        stack = [(root, xcenter, vert_loc, available_width)]
        
        while stack:
            curr_node, curr_x, curr_y, curr_width = stack.pop()
            pos[curr_node] = (curr_x, curr_y)
            
            children = list(G.successors(curr_node))
            if children:
                total_leaves = sum(leaf_counts.get(c, 1) for c in children)
                left_edge = curr_x - curr_width / 2
                
                # Pre-calculate to ensure left-to-right drawing order matches standard recursion
                children_params = []
                for child in children:
                    child_leaves = leaf_counts.get(child, 1)
                    child_width = curr_width * (child_leaves / total_leaves) if total_leaves > 0 else 0
                    child_xcenter = left_edge + child_width / 2
                    
                    children_params.append((child, child_xcenter, curr_y - vert_gap, child_width))
                    left_edge += child_width
                
                # Reverse before extending so the left-most child is popped off the stack first
                stack.extend(reversed(children_params))
                
        return pos

    roots = [n for n, d in G.in_degree() if d == 0]
    pos = {}
    x_offset = 0.0

    for root in roots:
        sub_pos = get_hierarchy_pos_iterative(G, root, xcenter=x_offset)
        pos.update(sub_pos)
        
        # Calculate root width to separate disconnected trees correctly
        root_width = max([p[0] for p in sub_pos.values()]) - min([p[0] for p in sub_pos.values()])
        x_offset += root_width + 2.0

    # ---------------------------------------------------------
    # 4. Geometry-Based Minimum Spacing Scaling (Bounded)
    # ---------------------------------------------------------
    total_leaves = sum(1 for n in G.nodes() if G.out_degree(n) == 0)
    max_depth = abs(min(y for x, y in pos.values())) if pos else 1
    
    min_space_per_level = 0.5   
    min_space_per_leaf = 0.8    
    
    desired_width = max(12.0, total_leaves * min_space_per_leaf)
    desired_height = max(12.0, max_depth * min_space_per_level)
    
    # Hard-cap dimensions to prevent exceeding Matplotlib's 2^16 pixel limit 
    # and to prevent system memory allocation crashes.
    fig_width = min(desired_width, 300.0)
    fig_height = min(desired_height, 300.0)

    plt.figure(figsize=(fig_width, fig_height), facecolor='white')
    ax = plt.gca()
    ax.axis('off')

    # ---------------------------------------------------------
    # 5. Determine Node Colors & Final Text Labels
    # ---------------------------------------------------------
    # Identify unique label texts applied to leaf nodes
    unique_leaf_labels = set()
    leaf_label_map = {}
    
    for node in G.nodes():
        if G.out_degree(node) == 0:
            texts = []
            if node in label_dict:
                texts.append(str(label_dict[node]).strip())
            if node in node_types:
                leaf_val = node_types[node]
                if pd.notna(leaf_val) and str(leaf_val).strip().lower() not in ['nan', 'none', 'null', '']:
                    texts.append(str(leaf_val).strip())
            
            if texts:
                combined = " ".join(texts)
                unique_leaf_labels.add(combined)
                leaf_label_map[node] = combined

    # Create a mapping from unique leaf label -> specific color using standard colormap
    try:
        cmap = plt.colormaps.get_cmap('tab10')
    except AttributeError:
        cmap = plt.cm.get_cmap('tab10') # Fallback for older matplotlib versions
        
    color_palette = [cmap(i % 10) for i in range(len(unique_leaf_labels))]
    label_color_map = dict(zip(sorted(list(unique_leaf_labels)), color_palette))

    # Assign final text and node color iteratively
    final_labels = {}
    node_colors = []
    
    for node in G.nodes():
        id_text = str(node) if draw_node_id else ""
        node_color = 'white' # Default internal/unlabeled node color
        
        if G.out_degree(node) == 0 and node in leaf_label_map:
            label_text = leaf_label_map[node]
            node_color = label_color_map[label_text] # Apply the unique color mapping
        else:
            # Handle intermediate branches that happen to have custom dictionary labels
            label_text = str(label_dict[node]).strip() if node in label_dict else ""
            
        # Stack the ID on top of the label with a newline character
        if id_text and label_text:
            final_labels[node] = f"{id_text}\n{label_text}"
        elif id_text:
            final_labels[node] = id_text
        elif label_text:
            final_labels[node] = label_text
            
        node_colors.append(node_color)

    # ---------------------------------------------------------
    # 6. Render Graph
    # ---------------------------------------------------------
    # Edges first so they sit cleanly under the nodes
    nx.draw_networkx_edges(G, pos, width=2.5, edge_color='black', arrows=False)
    
    # Nodes filled with dynamically mapped colors
    nx.draw_networkx_nodes(G, pos, node_size=80, node_color=node_colors, edgecolors='black', linewidths=1.5)

    if final_labels:
        width_span = max(x for x, y in pos.values()) - min(x for x, y in pos.values()) if len(pos) > 1 else 1
        x_shift = width_span * 0.005 
        
        pos_labels = {k: (v[0] + x_shift, v[1]) for k, v in pos.items()}

        nx.draw_networkx_labels(
            G, 
            pos_labels, 
            labels=final_labels, 
            font_size=9, 
            font_color='darkred',
            font_weight='bold',
            horizontalalignment='left',
            verticalalignment='center'
        )

    # Automatically generate a legend if colored labels exist
    if label_color_map:
        handles = [mlines.Line2D([], [], color='white', marker='o', markerfacecolor=color, 
                                 markersize=10, markeredgecolor='black', label=lbl) 
                   for lbl, color in label_color_map.items()]
        # Position legend in the upper right outside of the immediate drawing area to prevent overlap
        plt.legend(handles=handles, title="Leaf Types", loc='upper right', fontsize=12, title_fontsize=14)

    plt.title("Annotated Neuron Topology", fontsize=18, pad=20)
    plt.tight_layout()
    plt.show()

In [3]:
############################################
############################################
def read_swc(swc_path) -> pd.DataFrame:
    """
    Custom function that reads swc file as csv
    swc_path: str -> swc file location.
    """
    
    swc_df = pd.read_csv(swc_path, 
                         comment='#', 
                         header=None, 
                         sep=r'\s+', 
                         names=["node_id", "swc_type", "x", "y", "z", "r", "parent"])
    
    swc_df["node_id"] = swc_df["node_id"].astype(int)
    swc_df["parent"] = swc_df["parent"].astype(int)

    return swc_df

---

In [4]:
# Paths
neuron_id = "720575940636351845"
path_main = os.path.join("output", "swc_analysis_test")
path_swc_raw = os.path.join("data", "input_swc", "sk_lod1_783_healed", f"{neuron_id}.swc")
path_swc_simple = "_palceholder"
path_swc_labels = os.path.join(path_main, "simplified_swc", f"{neuron_id}.csv")
path_swc_final = os.path.join(path_main, "output_results", "simplified_swc", f"{neuron_id}.csv")


---

In [5]:
# Plot regular SWC neuron
df_swc_raw = read_swc(path_swc_raw)
df_swc_raw.head(10)

,node_id,swc_type,x,y,z,r,parent
0,1,1,615254.80,201704.33,192449.250,2989,-1
1,2,0,640636.70,160671.84,46255.707,157,25885
2,3,0,617016.40,205077.30,193770.030,337,24132
3,4,0,640585.40,134763.56,69088.740,137,25881
4,5,0,610358.00,200097.67,191602.660,0,23343
5,6,0,612042.70,198727.00,191109.690,126,23440
6,7,5,611879.80,198007.14,190725.080,187,23440
7,8,0,611548.94,197987.12,190677.980,63,7
8,9,0,611996.90,197973.72,190715.250,111,7
9,10,0,610999.75,198789.73,190279.160,57,23379


In [6]:
plot_neuron_topology(data = df_swc_raw)

ValueError: Image size of 86640x160300 pixels is too large. It must be less than 2^16 in each direction.

Error in callback <function _draw_all_if_interactive at 0x00000204EA2A5BC0> (for post_execute), with arguments args (),kwargs {}:


ValueError: Image size of 86640x160300 pixels is too large. It must be less than 2^16 in each direction.

ValueError: Image size of 86640x160300 pixels is too large. It must be less than 2^16 in each direction.

<Figure size 86640x160300 with 1 Axes>

---

In [ ]:
# Plot simplified SWC neuron.
df_swc_simple = simplify_swc_topology(swc_input=df_swc_raw, 
                                      swc_name=neuron_id,
                                      output_path="",
                                      save_csv=False)
df_swc_simple.head(10)

In [ ]:
plot_neuron_topology(data = df_swc_simple)

---

In [ ]:
# Plot simplified SWC neuron + node labels
df_swcl = pd.read_csv(path_swc_labels, index_col=0)
df_swcl_labels = df_swcl[["node_id", "type"]]

df_swcl.head(10)

In [ ]:
plot_neuron_topology(data = df_swcl,
                     labels_df = df_swcl_labels)

---

In [ ]:
# Plot simplified SWC neuron + node labels + clumpiness scores
df_swcf = pd.read_csv(path_swc_final)
df_swcf.head(10)

In [ ]:
plot_neuron_topology(data = df_swcf)